In [1]:
# ============================================================
#  07 Parks / Campuses / Commercial / Industrial Compounds
#  从 PBF 提取四类封闭/半封闭区域 polygon
#  计算形态指标 → 供 task typology 使用
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import osmium
import matplotlib.pyplot as plt
from shapely.geometry import Polygon, MultiPolygon, LineString
from shapely.prepared import prep as shapely_prep
from shapely.ops import unary_union
import warnings
warnings.filterwarnings("ignore")

OUT = Path("data_out")
OUT.mkdir(exist_ok=True)

PBF = Path("../04 Transport/data_raw/guangdong.osm.pbf")
BOUNDARY = Path("../04 Transport/data_raw/shenzhen_boundary.geojson")
EDGES_04 = Path("../04 Transport/data_out/sz_drive_edges.gpkg")
BUILDINGS_06 = Path("../06 Buildings/data_out/sz_buildings.gpkg")

shenzhen = gpd.read_file(BOUNDARY).to_crs(4326)
shenzhen_geom = shenzhen.union_all() if hasattr(shenzhen, "union_all") else shenzhen.unary_union
bbox = shenzhen_geom.bounds

print(f"PBF: {PBF.exists()}")
print(f"04 edges: {EDGES_04.exists()}")
print(f"06 buildings: {BUILDINGS_06.exists()}")

PBF: True
04 edges: True
06 buildings: True


In [2]:
# ============================================================
#  1. 从 PBF 提取四类 compound polygon
#  用 pyosmium area() 回调解析 multipolygon relations + closed ways
# ============================================================

# 分类规则: OSM tag → compound_type
TAG_RULES = [
    # (tag_key, tag_values, compound_type)
    ("leisure",  {"park", "nature_reserve", "recreation_ground"}, "park"),
    ("leisure",  {"garden"},                                      "park"),
    ("amenity",  {"school", "university", "college", "kindergarten"}, "campus"),
    ("landuse",  {"commercial", "retail"},                        "commercial"),
    ("amenity",  {"marketplace"},                                 "commercial"),
    ("landuse",  {"industrial"},                                  "industrial"),
]

class CompoundHandler(osmium.SimpleHandler):
    def __init__(self, minx, miny, maxx, maxy):
        super().__init__()
        self.minx, self.miny, self.maxx, self.maxy = minx, miny, maxx, maxy
        self.features = []

    def _classify(self, tags):
        for key, values, ctype in TAG_RULES:
            v = tags.get(key)
            if v and v in values:
                return ctype
        return None

    def area(self, a):
        tags = dict(a.tags)
        ctype = self._classify(tags)
        if ctype is None:
            return
        try:
            outer = a.outer_rings()
        except Exception:
            return
        for ring in outer:
            coords = [(n.lon, n.lat) for n in ring if n.location.valid()]
            if len(coords) < 4:
                continue
            lons = [c[0] for c in coords]
            lats = [c[1] for c in coords]
            if max(lons) < self.minx or min(lons) > self.maxx:
                continue
            if max(lats) < self.miny or min(lats) > self.maxy:
                continue
            try:
                poly = Polygon(coords)
                if not poly.is_valid:
                    poly = poly.buffer(0)
                if poly.is_empty or poly.area <= 0:
                    continue
                self.features.append({
                    "compound_type": ctype,
                    "name": tags.get("name", ""),
                    "osm_leisure": tags.get("leisure", ""),
                    "osm_amenity": tags.get("amenity", ""),
                    "osm_landuse": tags.get("landuse", ""),
                    "geometry": poly,
                })
            except Exception:
                pass

print("Extracting compounds from PBF ...")
ch = CompoundHandler(*bbox)
ch.apply_file(str(PBF), locations=True)
print(f"  bbox-filtered: {len(ch.features)} polygons")

if ch.features:
    compounds = gpd.GeoDataFrame(ch.features, crs=4326)
    compounds = gpd.clip(compounds, shenzhen_geom)
else:
    compounds = gpd.GeoDataFrame()

print(f"  clipped: {len(compounds)} compounds")
print(f"\n  By type:")
print(compounds["compound_type"].value_counts().to_string())
del ch

Extracting compounds from PBF ...
  bbox-filtered: 10686 polygons
  clipped: 7021 compounds

  By type:
compound_type
industrial    2645
campus        1541
commercial    1539
park          1296


In [3]:
# ============================================================
#  2. 形态指标计算
#  area / perimeter / compactness / enclosure_ratio / internal_road_density
# ============================================================

# 投影到 UTM 50N 计算面积和周长
compounds_proj = compounds.to_crs(32650)

compounds["area_m2"] = compounds_proj.geometry.area
compounds["perimeter_m"] = compounds_proj.geometry.length

# 紧凑度: 4π × area / perimeter² (圆=1, 越小越不规则)
compounds["compactness"] = (
    4 * np.pi * compounds["area_m2"] / (compounds["perimeter_m"] ** 2)
).clip(0, 1)

# 过滤掉过小的 polygon (< 500 m², 可能是标注噪声)
before = len(compounds)
compounds = compounds[compounds["area_m2"] >= 500].reset_index(drop=True)
print(f"Filtered: {before} -> {len(compounds)} (removed area < 500 m²)")

# ── 围合度: compound 边界上被建筑覆盖的比例 ──
print("\nCalculating enclosure ratio (building overlap on boundary) ...")

if BUILDINGS_06.exists():
    bldg = gpd.read_file(BUILDINGS_06)
    bldg_union = unary_union(bldg.geometry)
    bldg_prep = shapely_prep(bldg_union)

    enclosure_ratios = []
    for idx, row in compounds.iterrows():
        boundary_line = row.geometry.boundary
        try:
            if bldg_prep.intersects(boundary_line):
                overlap = boundary_line.intersection(bldg_union)
                ratio = overlap.length / boundary_line.length if boundary_line.length > 0 else 0
            else:
                ratio = 0
        except Exception:
            ratio = 0
        enclosure_ratios.append(min(ratio, 1.0))
    compounds["enclosure_ratio"] = enclosure_ratios
    del bldg, bldg_union, bldg_prep
    print(f"  Done. Mean enclosure: {compounds['enclosure_ratio'].mean():.3f}")
else:
    compounds["enclosure_ratio"] = np.nan
    print("  Skipped (06 buildings not found)")

# ── 内部路网密度: compound 内道路总长 / 面积 ──
print("\nCalculating internal road density ...")

if EDGES_04.exists():
    edges = gpd.read_file(EDGES_04)
    edges_proj = edges.to_crs(32650)

    road_densities = []
    compounds_proj2 = compounds.to_crs(32650)
    for idx, row in compounds_proj2.iterrows():
        poly = row.geometry
        try:
            clipped = gpd.clip(edges_proj, poly)
            total_len = clipped.geometry.length.sum()
            density = total_len / row.geometry.area if row.geometry.area > 0 else 0
        except Exception:
            density = 0
        road_densities.append(density)
    compounds["internal_road_density"] = road_densities
    del edges, edges_proj
    print(f"  Done. Mean road density: {compounds['internal_road_density'].mean():.4f} m/m²")
else:
    compounds["internal_road_density"] = np.nan
    print("  Skipped (04 edges not found)")

print(f"\nFinal compounds: {len(compounds)}")

Filtered: 7021 -> 6844 (removed area < 500 m²)

Calculating enclosure ratio (building overlap on boundary) ...
  Done. Mean enclosure: 0.158

Calculating internal road density ...
  Done. Mean road density: 0.0062 m/m²

Final compounds: 6844


In [4]:
# ============================================================
#  3. 保存输出 + 统计摘要
# ============================================================

compounds.to_file(OUT / "sz_compounds.gpkg", driver="GPKG")
print(f"Compounds -> data_out/sz_compounds.gpkg  ({len(compounds):,} rows)")

print("\n=== Summary by Type ===")
summary = compounds.groupby("compound_type").agg(
    count=("area_m2", "count"),
    total_area_km2=("area_m2", lambda x: x.sum() / 1e6),
    avg_area_m2=("area_m2", "mean"),
    avg_compactness=("compactness", "mean"),
    avg_enclosure=("enclosure_ratio", "mean"),
    avg_road_density=("internal_road_density", "mean"),
).round(3)
print(summary.to_string())

print("\n=== Top 10 Largest Compounds ===")
top10 = compounds.nlargest(10, "area_m2")[["name", "compound_type", "area_m2", "compactness", "enclosure_ratio"]]
top10["area_ha"] = (top10["area_m2"] / 10000).round(1)
print(top10[["name", "compound_type", "area_ha", "compactness", "enclosure_ratio"]].to_string())

print("\n=== High-Enclosure Compounds (>0.3) ===")
high_enc = compounds[compounds["enclosure_ratio"] > 0.3]
print(f"  {len(high_enc)} compounds with enclosure > 0.3")
print(high_enc["compound_type"].value_counts().to_string())

Compounds -> data_out/sz_compounds.gpkg  (6,844 rows)

=== Summary by Type ===
               count  total_area_km2  avg_area_m2  avg_compactness  avg_enclosure  avg_road_density
compound_type                                                                                      
campus          1540          38.263    24845.922            0.693          0.158             0.003
commercial      1533          37.073    24183.229            0.688          0.155             0.012
industrial      2642         136.317    51596.247            0.677          0.204             0.006
park            1129         112.242    99416.953            0.531          0.058             0.003

=== Top 10 Largest Compounds ===
           name compound_type  area_ha  compactness  enclosure_ratio
2338    凤凰山森林公园          park   2437.2     0.100695         0.010886
4123     罗田森林公园          park    444.9     0.255020         0.000000
4250        盐田港    industrial    416.2     0.426575         0.000000
142        